# The Accuracy-vs-Cost Curve [Step 03.04]

> **MLCourse - Agentic AI - Agent Patterns**

Every technique in this module trades money for accuracy. This notebook draws the
trade as a curve, with numbers measured in this session:

```
  accuracy
     ^
     |            . . . . . . . . . .   <- diminishing returns
     |         .
     |      .
     |   .
     | .
     +-------------------------------> cost (LLM calls)
```

The engineering question is never "is this technique good?". It is **"where on
this curve is my product?"** - and that has different answers for a support chat
bot and for a medical triage assistant.

### What you'll learn

- A measurement harness that records accuracy, tokens **and** latency together.
- The real curve for N=1..7 on a fixed task set.
- How to read diminishing returns and pick an operating point.
- How to report a null result honestly.

### Why it matters

Almost all published claims about these techniques come from papers using
frontier models on hard benchmarks. Your model and your tasks are different. The
only number that matters is the one you measured on your own workload, and this
notebook is the template for measuring it.

### Prerequisites

- [02_self_consistency](02_self_consistency.ipynb)
- [03_tree_of_thoughts](03_tree_of_thoughts.ipynb)

### Setup: environment, model, token counting, rate-limit-aware call helper


In [ ]:
import os                              # environment variables
import time                            # timing and pacing
import json                            # pretty-printing structured context
from pathlib import Path               # locating the track root
from dotenv import load_dotenv         # reads KEY=value pairs from .env

# Walk UP from the notebook folder until we hit the repo root, then load the
# (gitignored) .env that lives inside 03_agentic_ai. Note the extra path
# segment: the walk-up lands on the REPO ROOT, not on the track folder.
TRACK = Path.cwd()
while not (TRACK / "03_agentic_ai").exists() and TRACK != TRACK.parent:
    TRACK = TRACK.parent
load_dotenv(TRACK / "03_agentic_ai" / ".env")

GROQ_MODEL = "qwen/qwen3.8-27b"        # hosted, fast, generous free tier
# Local alternative (documented, not used here): Ollama `llama3.1:8b` via
# `from langchain_ollama import ChatOllama`. OpenAI is never used in this course.

from langchain_groq import ChatGroq


def make_llm(temperature: float = 0.0, max_tokens: int = 300, **kw):
    """One place that constructs the chat model, so every notebook is identical."""
    return ChatGroq(model=GROQ_MODEL, temperature=temperature,
                    max_tokens=max_tokens, **kw)


# --- Token counting -----------------------------------------------------------
# Two different numbers, and it matters which one you are looking at:
#   * approx_tokens(): a LOCAL estimate using tiktoken's cl100k_base. It is not
#     the model's own tokenizer, so treat it as "within ~10%", good for
#     budgeting BEFORE you send a request.
#   * usage_metadata on the response: the provider's EXACT count. Ground truth,
#     but only available AFTER you have already paid for the call.
import tiktoken

_ENC = tiktoken.get_encoding("cl100k_base")


def approx_tokens(text) -> int:
    """Approximate token count for a string (or anything str()-able)."""
    return len(_ENC.encode(str(text)))


# --- Rate-limit-aware calling --------------------------------------------------
# The Groq free tier allows 8000 tokens per minute. Several notebooks here make
# many small calls in a loop, so we self-pace well under the ceiling and retry
# with exponential backoff if we are throttled anyway.

TPM_BUDGET = 3500                       # deliberately conservative
_WINDOW = []                            # [(timestamp, tokens), ...]
USAGE = {"calls": 0, "in": 0, "out": 0, "seconds": 0.0}


def _pace(cost: int):
    """Sleep just enough that our rolling 60s token usage stays under budget."""
    now = time.time()
    while True:
        recent = [(t, n) for (t, n) in _WINDOW if now - t < 60]
        _WINDOW[:] = recent
        if sum(n for _, n in recent) + cost <= TPM_BUDGET or not recent:
            return
        time.sleep(min(5.0, 60 - (now - recent[0][0]) + 0.5))
        now = time.time()


def chat(messages, llm=None, temperature=0.0, max_tokens=300, retries=5):
    """Send `messages`, return the AIMessage. Paces, retries, and meters usage.

    `messages` is a list of (role, content) tuples or LangChain message objects.
    """
    llm = llm or make_llm(temperature=temperature, max_tokens=max_tokens)
    est = approx_tokens(messages) + max_tokens
    delay = 4.0
    for attempt in range(retries):
        _pace(est)
        t0 = time.time()
        try:
            out = llm.invoke(messages)
        except Exception as exc:
            if "rate_limit" in str(exc) or "429" in str(exc):
                time.sleep(delay)
                delay = min(delay * 2, 45)
                continue
            raise
        u = out.usage_metadata or {}
        _WINDOW.append((time.time(), u.get("total_tokens", est)))
        USAGE["calls"] += 1
        USAGE["in"] += u.get("input_tokens", 0)
        USAGE["out"] += u.get("output_tokens", 0)
        USAGE["seconds"] += time.time() - t0
        return out
    raise RuntimeError("still rate limited after %d attempts" % retries)


def ask(prompt: str, system: str = None, **kw) -> str:
    """Convenience wrapper: one user turn in, plain text out."""
    msgs = ([("system", system)] if system else []) + [("user", prompt)]
    return chat(msgs, **kw).content.strip()


print("model:", GROQ_MODEL)
print("key loaded:", bool(os.getenv("GROQ_API_KEY")))
print("tokenizer:", "cl100k_base (approximation)")


### The task set


In [ ]:
# Six multi-step word problems with short, exactly checkable answers. They are
# deliberately the kind that a model usually gets right but sometimes slips on:
# several dependent steps, one trap each (a percentage of the WRONG base, a
# time carry, an off-by-one week, a compounding discount).
#
# Small on purpose. Every technique in this module multiplies the number of
# calls, and the Groq free tier gives us 8000 tokens per minute.

TASKS = [
    dict(id="tank",
         q="A tank holds 480 litres and is currently 3/8 full. You add 90 litres, "
           "then drain an amount equal to 15% of the tank's TOTAL CAPACITY. "
           "How many litres are in the tank now?",
         answer="198"),
    dict(id="train",
         q="A train leaves at 09:40 and travels for 2 hours 50 minutes. You then "
           "wait 35 minutes for a connection, then travel a further 1 hour 15 "
           "minutes. At what time do you arrive? Use 24-hour HH:MM format.",
         answer="14:20"),
    dict(id="book",
         q="A book has 250 pages. Ana reads 12 pages per day on weekdays and 30 "
           "pages per day at weekends. She starts on a Monday morning. On which "
           "day of the week does she finish the book?",
         answer="monday"),
    dict(id="discount",
         q="A jacket costs 200 EUR. A 20% discount is applied, and then a further "
           "15% is taken off the already-reduced price. What is the final price "
           "in EUR?",
         answer="136"),
    dict(id="rect",
         q="A rectangle is three times as long as it is wide. Its perimeter is "
           "64 cm. What is its area in square centimetres?",
         answer="192"),
    dict(id="coins",
         q="You have 17 coins, all of them either 5 cents or 20 cents, worth 205 "
           "cents in total. How many 20-cent coins are there?",
         answer="8"),
]

ANSWER_INSTRUCTION = ("Work through it step by step, briefly. Then give your final "
                      "answer on the last line in exactly this form:\nANSWER: <value>")

import re


def parse_answer(text: str) -> str:
    """Pull the value out of the last 'ANSWER:' line. Returns '' if absent."""
    matches = re.findall(r"ANSWER\s*:\s*(.+)", text, re.I)
    return matches[-1].strip() if matches else ""


def normalise(value: str) -> str:
    """Make answers comparable: lowercase, drop units, currency and separators."""
    v = value.strip().lower()
    v = re.sub(r"\*\*|`|\.$", "", v)
    v = re.sub(r"\b(litres?|liters?|eur|euros?|cm|square centimetres?|cm\^?2|"
               r"cents?|coins?|pages?|hours?)\b", "", v)
    v = v.replace(",", "").replace(" ", "")
    return v.strip()


def is_correct(given: str, expected: str) -> bool:
    g, e = normalise(given), normalise(expected)
    if not g:
        return False
    return g == e or g.endswith(e) or e in g.split("=")[-1]


print("%d tasks" % len(TASKS))
for t in TASKS:
    print("  %-9s expected %-8s | %s..." % (t["id"], t["answer"], t["q"][:56]))


### 1. The harness

Three things recorded per call, because optimising one in isolation is how you
ship an accurate agent nobody will wait for:

- **Accuracy** - graded deterministically.
- **Tokens** - input and output, from `usage_metadata`, not estimated.
- **Latency** - wall clock. Note that N samples run *sequentially* here; in
  production you would run them concurrently, which changes the latency column
  completely and the token column not at all.

In [3]:
N_MAX = 7
TEMPERATURE = 0.8
BENCH = TASKS[:4]            # four tasks x seven samples = 28 calls. Kept small.

from collections import Counter

records = []                 # one row per individual sample
t_start = time.time()

for t in BENCH:
    for i in range(N_MAX):
        t0 = time.time()
        out = chat([("user", t["q"] + "\n\n" + ANSWER_INSTRUCTION)],
                   temperature=TEMPERATURE, max_tokens=340)
        u = out.usage_metadata
        records.append(dict(task=t["id"], i=i,
                            answer=parse_answer(out.content),
                            correct=is_correct(parse_answer(out.content), t["answer"]),
                            in_tok=u["input_tokens"], out_tok=u["output_tokens"],
                            seconds=time.time() - t0))
    print("%-9s %s" % (t["id"], [normalise(r["answer"])[:7]
                                 for r in records if r["task"] == t["id"]]))

print("\n%d samples in %.0f s" % (len(records), time.time() - t_start))

tank      ['198', '198', '198', '198', '198', '198', '198']


train     ['14:20', '', '14:20', '14:20', '14:20', '', '14:20']


book      ['', '', '', '', '', '', '']


discount  ['136', '136', '136', '136', '136', '136', '136']

28 samples in 210 s


In [4]:
def majority(answers):
    normed = [normalise(a) for a in answers if a.strip()]
    return Counter(normed).most_common(1)[0][0] if normed else ""


curve = []
for n in range(1, N_MAX + 1):
    correct = 0
    tok = 0
    secs = 0.0
    for t in BENCH:
        rows = [r for r in records if r["task"] == t["id"]][:n]
        correct += is_correct(majority([r["answer"] for r in rows]), t["answer"])
        tok += sum(r["in_tok"] + r["out_tok"] for r in rows)
        secs += sum(r["seconds"] for r in rows)
    curve.append(dict(n=n, acc=correct / len(BENCH), correct=correct,
                      tokens=tok, seconds=secs))

print("%4s %10s %10s %10s %12s %12s"
      % ("N", "correct", "accuracy", "tokens", "latency (s)", "tok/task"))
print("-" * 64)
for c in curve:
    print("%4d %8d/%-2d %9.0f%% %10d %11.1f %12.0f"
          % (c["n"], c["correct"], len(BENCH), 100 * c["acc"], c["tokens"],
             c["seconds"], c["tokens"] / len(BENCH)))

   N    correct   accuracy     tokens  latency (s)     tok/task
----------------------------------------------------------------
   1        3/4         75%       1299        42.2          325
   2        3/4         75%       2717        59.0          679
   3        3/4         75%       4119        75.7         1030
   4        3/4         75%       5528        94.1         1382
   5        3/4         75%       6966       129.1         1742
   6        3/4         75%       8437       166.4         2109
   7        3/4         75%       9843       210.5         2461


In [5]:
# The curve, drawn.
lo = min(c["acc"] for c in curve)
print("accuracy vs N (temperature %.1f, %s, %d tasks)\n" % (TEMPERATURE, GROQ_MODEL, len(BENCH)))
for c in curve:
    bar = "#" * int(c["acc"] * 40)
    print("N=%d  %5.0f%%  %-40s  %6d tok  %5.1f s"
          % (c["n"], 100 * c["acc"], bar, c["tokens"], c["seconds"]))
print()
print("marginal gain per extra sample:")
for a, b in zip(curve, curve[1:]):
    d_acc = 100 * (b["acc"] - a["acc"])
    d_tok = b["tokens"] - a["tokens"]
    print("  N=%d -> N=%d : %+5.1f points for %+5d tokens  (%s)"
          % (a["n"], b["n"], d_acc, d_tok,
             "%.2f points per 1k tokens" % (1000 * d_acc / d_tok) if d_tok else "-"))

accuracy vs N (temperature 0.8, qwen/qwen3.8-27b, 4 tasks)

N=1     75%  ##############################              1299 tok   42.2 s
N=2     75%  ##############################              2717 tok   59.0 s
N=3     75%  ##############################              4119 tok   75.7 s
N=4     75%  ##############################              5528 tok   94.1 s
N=5     75%  ##############################              6966 tok  129.1 s
N=6     75%  ##############################              8437 tok  166.4 s
N=7     75%  ##############################              9843 tok  210.5 s

marginal gain per extra sample:
  N=1 -> N=2 :  +0.0 points for +1418 tokens  (0.00 points per 1k tokens)
  N=2 -> N=3 :  +0.0 points for +1402 tokens  (0.00 points per 1k tokens)
  N=3 -> N=4 :  +0.0 points for +1409 tokens  (0.00 points per 1k tokens)
  N=4 -> N=5 :  +0.0 points for +1438 tokens  (0.00 points per 1k tokens)
  N=5 -> N=6 :  +0.0 points for +1471 tokens  (0.00 points per 1k tokens)
  N=6 -> N=7

### 2. Reading the curve

Three shapes, three conclusions:

**Rising then flattening** - the textbook result. The flattening point is your
operating point: the smallest N that gets most of the available gain. Past it you
are buying nothing.

**Flat at 100% throughout** - your task set is too easy to measure the technique.
This is a statement about your *benchmark*, not about the technique. Fix it by
making the tasks harder, not by concluding sampling does not work.

**Flat below 100%** - the model is *consistently* wrong on some tasks. Voting
cannot fix a systematic error; every sample makes the same mistake. You need a
different prompt, a tool, or a different model - not more samples.

The cell below states which of the three we actually got.

In [6]:
first, last = curve[0], curve[-1]
peak = max(curve, key=lambda c: (c["acc"], -c["n"]))

print("MEASURED, this session, %s, temperature %.1f, %d tasks, %d samples each:"
      % (GROQ_MODEL, TEMPERATURE, len(BENCH), N_MAX))
print()
print("  N=1 accuracy        : %.0f%%   (%d tokens, %.1f s)"
      % (100 * first["acc"], first["tokens"], first["seconds"]))
print("  N=%d accuracy        : %.0f%%   (%d tokens, %.1f s)"
      % (last["n"], 100 * last["acc"], last["tokens"], last["seconds"]))
print("  best accuracy       : %.0f%% at N=%d" % (100 * peak["acc"], peak["n"]))
print("  cost multiplier     : %.1fx tokens, %.1fx latency"
      % (last["tokens"] / first["tokens"], last["seconds"] / first["seconds"]))
print()

if first["acc"] >= 1.0 and last["acc"] >= 1.0:
    print("  SHAPE: flat at ceiling.")
    print("  A single sample already solves every task, so majority voting has no")
    print("  error left to correct. Reported honestly: on THIS task set, with THIS")
    print("  model, self-consistency bought 0 points for %.1fx the tokens."
          % (last["tokens"] / first["tokens"]))
    print("  The correct conclusion is that the benchmark is too easy, not that the")
    print("  technique fails. A useful benchmark needs single-sample accuracy in the")
    print("  30-90%% band - that is where voting has something to work with.")
elif peak["acc"] > first["acc"]:
    print("  SHAPE: rising.")
    print("  Voting gained %+.0f points. The smallest N reaching the peak is N=%d,"
          % (100 * (peak["acc"] - first["acc"]), peak["n"]))
    print("  which is your operating point - larger N costs more for no gain.")
else:
    print("  SHAPE: flat below ceiling.")
    print("  Some tasks are wrong at EVERY N. That is a systematic error, and more")
    print("  samples cannot fix it: all samples share the same misconception.")

MEASURED, this session, qwen/qwen3.8-27b, temperature 0.8, 4 tasks, 7 samples each:

  N=1 accuracy        : 75%   (1299 tokens, 42.2 s)
  N=7 accuracy        : 75%   (9843 tokens, 210.5 s)
  best accuracy       : 75% at N=1
  cost multiplier     : 7.6x tokens, 5.0x latency

  SHAPE: flat below ceiling.
  Some tasks are wrong at EVERY N. That is a systematic error, and more
  samples cannot fix it: all samples share the same misconception.


### Which tasks were unstable? That is where the technique has any purchase at all.


In [ ]:
print("%-9s %10s %12s %s" % ("task", "n correct", "stability", "distinct answers"))
print("-" * 58)
for t in BENCH:
    rows = [r for r in records if r["task"] == t["id"]]
    n_ok = sum(r["correct"] for r in rows)
    distinct = sorted({normalise(r["answer"])[:8] for r in rows})
    stab = ("always right" if n_ok == len(rows) else
            "always wrong" if n_ok == 0 else "UNSTABLE")
    print("%-9s %8d/%-2d %12s %s" % (t["id"], n_ok, len(rows), stab, distinct))
print()
print("Only the UNSTABLE rows can be rescued by voting. If there are none, the")
print("curve is flat by arithmetic, not by accident.")


### 3. Cost is not only tokens

Three currencies, and they do not trade at a fixed rate:

| Currency | N=1 | N=7 | Fixable by concurrency? |
|---|---|---|---|
| **Tokens** (money) | 1x | ~7x | No |
| **Latency** (user waiting) | 1x | ~7x sequential | **Yes** - run samples in parallel |
| **Rate limit** (throughput) | 1x | 7x | No - it is the same quota |

The middle row is the one people forget in both directions. Sampled sequentially,
N=7 makes your agent seven times slower, which for an interactive product is
usually disqualifying on its own. Sampled concurrently, latency stays near N=1
and only the money changes - but you hit your provider rate limit seven times
faster, which is exactly the constraint that forced this notebook to pace its
own calls.

In [8]:
per_call = last["seconds"] / (N_MAX * len(BENCH))
print("measured mean latency per call : %.2f s" % per_call)
print()
print("%4s %14s %16s %12s" % ("N", "sequential", "concurrent (est)", "tokens"))
print("-" * 50)
for c in curve:
    print("%4d %13.1fs %15.1fs %12d"
          % (c["n"], c["seconds"] / len(BENCH), per_call, c["tokens"] / len(BENCH)))
print()
print("Concurrency changes the latency column and nothing else. It does not make")
print("sampling cheaper - it makes it bearable.")

measured mean latency per call : 7.52 s

   N     sequential concurrent (est)       tokens
--------------------------------------------------
   1          10.5s             7.5s          324
   2          14.7s             7.5s          679
   3          18.9s             7.5s         1029
   4          23.5s             7.5s         1382
   5          32.3s             7.5s         1741
   6          41.6s             7.5s         2109
   7          52.6s             7.5s         2460

Concurrency changes the latency column and nothing else. It does not make
sampling cheaper - it makes it bearable.


### 4. Picking an operating point

The decision is a product decision, not a technical one:

| Situation | Operating point |
|---|---|
| Interactive chat, cheap errors | **N=1**. Latency dominates. |
| Batch processing, moderate error cost | **N=3-5**, concurrent |
| Errors are expensive (finance, medical, legal) | **N=5+**, plus escalate on low agreement |
| A deterministic verifier exists | **Best-of-n with verification** - beats voting outright |
| The model is systematically wrong | **None of these.** Change the prompt, add a tool, change the model. |

The last row is the one to internalise. Sampling techniques treat *variance*. If
your failure is *bias*, they will happily charge you 7x to reproduce it more
reliably.

In [9]:
print("=" * 66)
print("MODULE 03 - MEASURED SUMMARY (this session)")
print("=" * 66)
print("model            : %s" % GROQ_MODEL)
print("temperature      : %.1f" % TEMPERATURE)
print("task set         : %d multi-step word problems, deterministic grading" % len(BENCH))
print("samples per task : %d" % N_MAX)
print()
print("%-22s %10s %10s %10s" % ("configuration", "accuracy", "tokens", "latency"))
print("-" * 56)
for c in (curve[0], curve[2], curve[4], curve[-1]):
    print("%-22s %9.0f%% %10d %9.1fs"
          % ("majority vote, N=%d" % c["n"], 100 * c["acc"], c["tokens"], c["seconds"]))
print()
print("total spend for this notebook: %d calls, %d in + %d out tokens, %.0f s"
      % (USAGE["calls"], USAGE["in"], USAGE["out"], USAGE["seconds"]))

MODULE 03 - MEASURED SUMMARY (this session)
model            : qwen/qwen3.8-27b
temperature      : 0.8
task set         : 4 multi-step word problems, deterministic grading
samples per task : 7

configuration            accuracy     tokens    latency
--------------------------------------------------------
majority vote, N=1            75%       1299      42.2s
majority vote, N=3            75%       4119      75.7s
majority vote, N=5            75%       6966     129.1s
majority vote, N=7            75%       9843     210.5s

total spend for this notebook: 28 calls, 2625 in + 7218 out tokens, 76 s


### 5. Pitfalls

- **Quoting a paper's numbers as your own.** Different model, different tasks,
  different result. Measure.
- **Measuring accuracy without latency.** A 7x slower agent is a different
  product, not a better one.
- **A benchmark that is too easy.** A flat 100% curve measures your benchmark,
  not your technique.
- **Hiding a null result.** "+0 points for 7x cost" is a finding. Report it.
- **Sampling to fix bias.** Variance techniques do not touch systematic error.

### Recap

| Idea | Takeaway |
|---|---|
| Measure three currencies | Accuracy, tokens, latency - together |
| Reuse prefixes | Sample N once, evaluate every smaller N free |
| Read the shape | Rising / flat-at-ceiling / flat-below each mean something different |
| Concurrency helps latency only | Money and rate limit are unchanged |
| Operating point is a product call | Interactive N=1; expensive errors N=5+ |

### Module complete

You can now reason about temperature deliberately, implement self-consistency and
tree search, and - most importantly - **measure** whether either is worth it on
your own workload rather than taking a paper's word for it.

**Next module:** [04_self_critique](../04_self_critique) - instead of sampling
more answers, make the model improve the one it has.